# Assignment 7 – MongoDB & Statistics

## Question 1: NoSQL vs Relational Databases

### a) Key Differences Between Relational Databases and MongoDB

| Dimension | Relational DB (MySQL/PostgreSQL) | MongoDB |
|---|---|---|
| **Data Model** | Fixed rows & columns with predefined schema | Flexible JSON-like documents (BSON) |
| **Schema** | Strict schema defined upfront; migrations needed | Schema-less / schema-flexible |
| **Relationships** | JOINs across normalized tables | Embedding or referencing by ObjectId |
| **Scaling** | Vertical (more CPU/RAM on one server) | Horizontal via sharding across many servers |
| **Query Language** | SQL | JSON-style query language + aggregation pipeline |
| **Transactions** | Robust ACID transactions across tables | Multi-document ACID from v4.0 (less performant) |

### b) When MongoDB is Preferred Over SQL

- Data structure is hierarchical or varies per record (e.g., product catalogs with different attributes per product).
- Application needs to scale horizontally for very high read/write throughput.
- Schema evolves frequently during rapid development, avoiding costly SQL migrations.
- Storing semi-structured data like user activity logs, IoT sensor data, or social media posts.
- Reads are frequent and data is naturally accessed as a unit — embedding eliminates expensive JOINs.

### c) Schema Flexibility and Its Risks

Schema flexibility means no two documents in the same collection need the same fields. A new field can be added without altering a schema or migrating existing records — powerful during early development.

**Risks:**
- **Inconsistency:** Different documents may store the same information in different formats (e.g., `'phone'` vs `'phone_number'` vs `'mobile'`).
- **Application-level bugs:** The database does not reject bad data — invalid or missing fields silently pass through.
- **Analytics difficulty:** Inconsistent fields make aggregation and reporting harder.
- **Mitigation:** Use JSON Schema validation at the collection level, and enforce structure via Pydantic or Mongoose models.

### d) Embedding vs Referencing and Performance Impact

| Strategy | Description | Best When |
|---|---|---|
| **Embedding** | Related data nested as a sub-document inside the parent | Data is always accessed together; nested data doesn't grow unboundedly |
| **Referencing** | Related data in a separate collection, linked by ObjectId | Data is large, shared across many documents, or independently updated |

> **Rule of thumb:** Embed for data accessed together and owned by one parent. Reference for large, shared, or independently updated data.

In [ ]:
# Embedding vs Referencing — Python / PyMongo illustration
from pymongo import MongoClient
from bson import ObjectId
from datetime import datetime

# client = MongoClient("mongodb://localhost:27017/")
# db = client["shop"]

# ── EMBEDDING example ──────────────────────────────────────────────────────
embedded_order = {
    "customer_name": "Rahul Sharma",
    "order_date": datetime.utcnow(),
    "items": [                               # items embedded inside order
        {"product": "Laptop", "qty": 1, "price": 45000},
        {"product": "Mouse",  "qty": 2, "price": 800},
    ],
    "total_amount": 46600,
    "status": "Pending"
}
print("Embedded document:")
import json
print(json.dumps({k: str(v) if not isinstance(v, (dict, list, int, float, str, bool)) else v
                  for k, v in embedded_order.items()}, indent=2))

# ── REFERENCING example ────────────────────────────────────────────────────
customer_doc = {"_id": ObjectId(), "name": "Rahul Sharma", "email": "rahul@example.com"}
referenced_order = {
    "customer_id": customer_doc["_id"],      # only the ObjectId reference
    "order_date": datetime.utcnow(),
    "total_amount": 46600,
    "status": "Pending"
}
print("\nReferenced order document:")
print({"customer_id": str(referenced_order["customer_id"]),
       "total_amount": referenced_order["total_amount"],
       "status": referenced_order["status"]})

## Question 2: Indexing, Sharding & Replication

### a) How Indexes Improve MongoDB Query Performance

Without an index, MongoDB performs a **collection scan** — reading every document to find matches (O(n)). An index is a B-tree storing sorted field values with pointers to documents, reducing lookup time to **O(log n)**.

- A single-field index on `email` allows instant user lookup.
- Indexes speed up sort operations — if data is already indexed in sorted order, MongoDB skips the sort step.
- **Trade-off:** Indexes consume disk space and slow down writes (every insert/update/delete must also update each index). Over-indexing is as harmful as under-indexing.

### b) Sharding — What It Is and When It Is Required

Sharding is MongoDB's **horizontal scaling** mechanism — it distributes data across multiple servers (shards), partitioned by a **shard key**. A query router (`mongos`) directs requests to the correct shard(s).

**Sharding is required when:**
- Dataset exceeds single-server storage capacity.
- Write throughput exceeds what one server can handle (e.g., millions of inserts/sec from IoT).
- Read operations need distribution for low latency at global scale.
- *Examples:* Social media platforms, real-time analytics, high-volume e-commerce.

### c) Replication and High Availability

MongoDB replication uses **Replica Sets** — typically 3 nodes: one **Primary** (accepts all writes) + **Secondaries** (replicate data asynchronously) + optional **Arbiter** (participates in elections, holds no data).

- **Automatic failover:** If the Primary goes down, remaining nodes elect a new Primary within 10–30 seconds — no manual intervention needed.
- **Secondaries** can serve reads (eventual consistency), distributing read load.
- A 3-node replica set survives the failure of **one** node without downtime.

### d) Impact of Improper Shard Key Selection

| Problem | Cause |
|---|---|
| **Hotspotting** | Monotonically increasing key (e.g., timestamp/auto-ID) routes all writes to one shard |
| **Scatter-gather queries** | Shard key doesn't match query patterns — every query hits all shards |
| **Uneven distribution** | Low-cardinality key (e.g., boolean) can create at most 2 chunks |
| **Irreversibility** | Changing a shard key requires expensive re-sharding of the entire collection |

> **Best practice:** Choose a shard key with **high cardinality**, **even distribution**, and **alignment with common query patterns**. Compound shard keys often outperform single-field keys.

In [ ]:
# Index creation and explain() plan — PyMongo examples

# Single-field descending index on order_date
# db.orders.create_index([("order_date", -1)])

# Compound index: equality field first (ESR rule), then sort/range field
# db.orders.create_index(
#     [("customer_id", 1), ("order_date", -1)],
#     name="idx_customer_date"
# )

# Verify index usage with explain()
# plan = db.orders.find(
#     {"customer_id": customer_id, "order_date": {"$gt": cutoff_date}}
# ).explain("executionStats")
#
# Key fields to inspect:
# plan["queryPlanner"]["winningPlan"]["stage"]        → should be "IXSCAN"
# plan["executionStats"]["totalDocsExamined"]         → should ≈ totalDocsReturned
# plan["executionStats"]["executionTimeMillis"]       → compare before/after index

# Listing all indexes
# db.orders.index_information()

print("Index management commands (run in PyMongo or MongoDB shell):")
index_examples = {
    "single_field": 'db.orders.create_index([("order_date", -1)])',
    "compound":     'db.orders.create_index([("customer_id", 1), ("order_date", -1)], name="idx_customer_date")',
    "list_indexes": 'db.orders.index_information()',
    "explain":      'db.orders.find({...}).explain("executionStats")',
}
for name, cmd in index_examples.items():
    print(f"  {name}: {cmd}")

## Question 3: Descriptive Statistics & Distributions

### a) Difference Between Mean, Median, and Variance

- **Mean (Average):** Sum of all values divided by n. Represents the centre of gravity of the data. Highly sensitive to outliers — one extreme value can pull the mean far from where most data lies.
- **Median:** The middle value when data is sorted ascending. Robust to outliers — a better measure of "typical" value for skewed data.
- **Variance:** Measures how spread out data points are around the mean. Formula: `Σ(xᵢ − μ)² / n`. A variance of 0 means all values are identical. **Standard deviation** = √variance, expressed in the same units as the original data.

### b) What High Variance Indicates in Customer Spending

High variance means spending amounts are spread very widely around the average:

- The customer base is **highly heterogeneous** — some spend ₹200, others ₹50,000, but the average (say ₹5,000) represents no typical customer.
- Marketing strategies based on the average miss both segments.
- **Customer segmentation is essential** — split into low / medium / high-value tiers.
- Revenue is often concentrated in a small number of high spenders (**Pareto Principle: 80% revenue from 20% of customers**). Losing top customers disproportionately harms the business.

### c) Normal Distribution vs Skewed Distribution

| Feature | Normal Distribution | Skewed Distribution |
|---|---|---|
| **Shape** | Symmetric bell curve | Asymmetric |
| **Mean / Median / Mode** | All equal, at centre | Differ; mean pulled toward tail |
| **Tails** | Symmetric on both sides | One tail extends further |
| **Examples** | Heights, IQ scores, measurement errors | Revenue, order value, session duration |

### d) Why Business Metrics Often Show Right-Skewed Distributions

- **Hard floor at zero:** A customer cannot spend a negative amount — this truncates the left tail.
- **No ceiling on the upside:** A single whale customer can spend 100× the average, creating a long right tail.
- **Most customers are moderate; a few are extreme:** power users or large accounts generate disproportionately high values.
- **Implication:** The mean overstates the "typical" customer. The **median is a more accurate** measure of typical behaviour. Always report both.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Simulate customer spending (right-skewed via log-normal)
spending = np.random.lognormal(mean=8, sigma=1.2, size=1000)

mean_val   = np.mean(spending)
median_val = np.median(spending)
var_val    = np.var(spending)
std_val    = np.std(spending)

print("=== Descriptive Statistics — Customer Spending ===")
print(f"Mean    : ₹{mean_val:>10,.2f}")
print(f"Median  : ₹{median_val:>10,.2f}")
print(f"Variance: {var_val:>14,.2f}")
print(f"Std Dev : ₹{std_val:>10,.2f}")

# Visualise
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Right-skewed spending distribution
axes[0].hist(spending, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(mean_val,   color='red',    linestyle='--', linewidth=2, label=f'Mean ₹{mean_val:,.0f}')
axes[0].axvline(median_val, color='green',  linestyle='--', linewidth=2, label=f'Median ₹{median_val:,.0f}')
axes[0].set_title('Customer Spending — Right-Skewed Distribution')
axes[0].set_xlabel('Monthly Spending (₹)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Normal distribution (for comparison)
x = np.linspace(-4, 4, 300)
axes[1].plot(x, stats.norm.pdf(x), color='darkorange', linewidth=2)
axes[1].fill_between(x, stats.norm.pdf(x), alpha=0.3, color='darkorange')
axes[1].set_title('Normal Distribution (Symmetric)')
axes[1].set_xlabel('Standard Deviations from Mean')
axes[1].set_ylabel('Probability Density')

plt.tight_layout()
plt.savefig('distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## Question 4: Random Variables & Bayes' Theorem

### a) What is a Random Variable?

A random variable is a variable whose value is determined by the outcome of a random experiment. It maps outcomes to numerical values. For example, if you roll a die, X = number shown. X can take values 1–6, each with probability 1/6. Random variables are the mathematical foundation of probability theory, allowing us to model uncertainty and compute expected values, variances, and probabilities.

### b) Discrete vs Continuous Random Variables

| Feature | Discrete | Continuous |
|---|---|---|
| **Values** | Countable distinct values (usually integers) | Any value in a range (including decimals) |
| **Function** | Probability Mass Function (PMF) | Probability Density Function (PDF) |
| **P(X = exact value)** | Well-defined | Always 0; probability defined for intervals |
| **Examples** | # customers in a store, # defects in a batch | Time on website, temperature, spending amount |

### c) Covariance and Its Interpretation

Covariance measures the **direction of the linear relationship** between two variables.

`Cov(X, Y) = (1/n) × Σ[(Xᵢ − μₓ)(Yᵢ − μᵧ)]`

| Sign | Meaning | Example |
|---|---|---|
| **Positive** | Both increase together | Marketing spend & revenue |
| **Negative** | One increases, other decreases | Price increases & demand |
| **Near zero** | No consistent linear relationship | Shoe size & purchase frequency |

> **Limitation:** Covariance is scale-dependent. **Pearson Correlation** (covariance ÷ product of std deviations) normalises it to [−1, +1] for easier interpretation.

### d) Bayes' Theorem in Business Decision-Making

`P(A|B) = [P(B|A) × P(A)] / P(B)`

- **P(A)** = prior probability (belief before new evidence)
- **P(B|A)** = likelihood (probability of evidence given hypothesis)
- **P(A|B)** = posterior probability (updated belief after evidence)

**Business Example — Fraud Detection:**
- Prior: P(Fraud) = 0.02 (2% of transactions are fraudulent)
- P(Flagged | Fraud) = 0.90 (90% of real fraud is caught)
- P(Flagged | Not Fraud) = 0.05 (5% false alarm rate)

Using Bayes, we compute the posterior probability that a flagged transaction is actually fraudulent — allowing the business to prioritize human review for the highest-risk cases rather than auto-blocking everything.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Covariance & Correlation ────────────────────────────────────────────────
np.random.seed(42)
marketing_spend = np.random.normal(50000, 10000, 100)
revenue = marketing_spend * 3.5 + np.random.normal(0, 20000, 100)

cov_matrix = np.cov(marketing_spend, revenue)
covariance  = cov_matrix[0, 1]
correlation = np.corrcoef(marketing_spend, revenue)[0, 1]

print("=== Covariance & Correlation ===")
print(f"Covariance  : {covariance:,.2f}  (scale-dependent)")
print(f"Correlation : {correlation:.4f}  (scale-free, range −1 to +1)")

# ── Bayes' Theorem — Fraud Detection ────────────────────────────────────────
p_fraud           = 0.02
p_not_fraud       = 1 - p_fraud
p_flagged_fraud   = 0.90
p_flagged_no_fraud = 0.05

p_flagged = (p_flagged_fraud * p_fraud) + (p_flagged_no_fraud * p_not_fraud)
p_fraud_given_flagged = (p_flagged_fraud * p_fraud) / p_flagged

print("
=== Bayes' Theorem — Fraud Detection ===")
print(f"P(Flagged)               : {p_flagged:.4f}")
print(f"P(Fraud | Flagged)       : {p_fraud_given_flagged:.4f}  ({p_fraud_given_flagged*100:.2f}%)")
print(f"Baseline P(Fraud)        : {p_fraud:.4f}  ({p_fraud*100:.2f}%)")
print(f"Uplift from flagging     : {p_fraud_given_flagged/p_fraud:.1f}×")

# ── Visualisation ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(marketing_spend, revenue, alpha=0.6, color='steelblue')
m, b = np.polyfit(marketing_spend, revenue, 1)
x_line = np.linspace(marketing_spend.min(), marketing_spend.max(), 100)
axes[0].plot(x_line, m * x_line + b, color='red', linewidth=2)
axes[0].set_title(f'Marketing Spend vs Revenue
(Corr = {correlation:.3f})')
axes[0].set_xlabel('Marketing Spend (₹)')
axes[0].set_ylabel('Revenue (₹)')

labels = ['P(Fraud)
Baseline', 'P(Fraud | Flagged)
Posterior']
values = [p_fraud, p_fraud_given_flagged]
colors = ['#90CAF9', '#EF9A9A']
bars = axes[1].bar(labels, values, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.2%}', ha='center', fontweight='bold')
axes[1].set_title("Bayes' Theorem — Fraud Probability Update")
axes[1].set_ylabel('Probability')
axes[1].set_ylim(0, 0.25)

plt.tight_layout()
plt.savefig('bayes_covariance.png', dpi=150, bbox_inches='tight')
plt.show()

## Question 5: MongoDB Queries for Orders Collection

### 1. Insert a New Order
Inserts a single document with embedded items array, total amount, and status.

In [ ]:
# ── 1. Insert a new order ───────────────────────────────────────────────────
# PyMongo equivalent of the MongoDB shell insertOne()

from datetime import datetime
# from pymongo import MongoClient
# from bson import ObjectId
# client = MongoClient("mongodb://localhost:27017/")
# db = client["shop"]

new_order = {
    "customer_id": "64a1f2b3c4d5e6f7a8b9c0d1",   # ObjectId in production
    "customer_name": "Rahul Sharma",
    "items": [
        {"product": "Laptop", "quantity": 1, "price": 45000},
        {"product": "Mouse",  "quantity": 2, "price": 800},
    ],
    "total_amount": 46600,
    "status": "Pending",
    "order_date": datetime.utcnow()
}

# result = db.orders.insert_one(new_order)
# print("Inserted ID:", result.inserted_id)
print("Order document to insert:")
import json
printable = {k: str(v) if isinstance(v, datetime) else v for k, v in new_order.items()}
print(json.dumps(printable, indent=2))

### 2. Find All Orders Above ₹5,000
`$gt` filters documents where `total_amount > 5000`. The projection returns only specified fields.

In [ ]:
# ── 2. Find orders above ₹5000 ──────────────────────────────────────────────
# db.orders.find(
#     {"total_amount": {"$gt": 5000}},
#     {"customer_name": 1, "total_amount": 1, "status": 1, "order_date": 1}
# ).sort("total_amount", -1)

# Demonstrate logic with sample data
import pandas as pd

sample_orders = pd.DataFrame([
    {"customer_name": "Rahul Sharma",  "total_amount": 46600, "status": "Pending",   "order_date": "2024-06-01"},
    {"customer_name": "Priya Mehta",   "total_amount": 3200,  "status": "Delivered",  "order_date": "2024-06-03"},
    {"customer_name": "Amit Singh",    "total_amount": 12000, "status": "Processing", "order_date": "2024-06-05"},
    {"customer_name": "Sneha Joshi",   "total_amount": 850,   "status": "Pending",    "order_date": "2024-06-07"},
    {"customer_name": "Vikram Nair",   "total_amount": 9500,  "status": "Delivered",  "order_date": "2024-06-08"},
])

result = sample_orders[sample_orders["total_amount"] > 5000].sort_values("total_amount", ascending=False)
print("Orders above ₹5,000 (sorted by amount desc):")
print(result.to_string(index=False))

### 3. Update Status to 'Delivered' for Orders Older Than 7 Days
`updateMany` + `$lt` + `$ne` targets orders older than 7 days that are not already delivered.

In [ ]:
# ── 3. Update old orders to 'Delivered' ─────────────────────────────────────
from datetime import datetime, timedelta

seven_days_ago = datetime.utcnow() - timedelta(days=7)

# PyMongo:
# db.orders.update_many(
#     {"order_date": {"$lt": seven_days_ago}, "status": {"$ne": "Delivered"}},
#     {"$set": {"status": "Delivered", "updated_at": datetime.utcnow()}}
# )

shell_cmd = (
    "const sevenDaysAgo = new Date();\n"
    "sevenDaysAgo.setDate(sevenDaysAgo.getDate() - 7);\n"
    "db.orders.updateMany(\n"
    "  { order_date: { $lt: sevenDaysAgo }, status: { $ne: 'Delivered' } },\n"
    "  { $set: { status: 'Delivered', updated_at: new Date() } }\n"
    ");"
)
print("MongoDB Shell equivalent:")
print(shell_cmd)

### 4. Delete Cancelled Orders
`deleteMany` removes all documents matching the filter. Always preview with `find()` before deleting in production.

In [ ]:
# ── 4. Delete cancelled orders ──────────────────────────────────────────────
# SAFETY: preview first
# cancelled_preview = list(db.orders.find({"status": "Cancelled"}, {"_id": 1, "customer_name": 1}))
# print(f"Documents to be deleted: {len(cancelled_preview)}")

# Then delete
# result = db.orders.delete_many({"status": "Cancelled"})
# print(f"Deleted {result.deleted_count} cancelled order(s).")

print("Safe deletion workflow:")
print("  Step 1: db.orders.find({'status': 'Cancelled'})        # preview")
print("  Step 2: db.orders.delete_many({'status': 'Cancelled'}) # confirm & delete")

## Question 6: Indexes in MongoDB

### Single-Field Index on `order_date`
`-1` creates a **descending** index — optimal for queries that filter or sort by date in descending order (most recent first), the most common pattern in order management.

### Compound Index for `customer_id` + `order_date`
Supports queries that filter by `customer_id` AND sort/filter by `order_date` (e.g., *"all orders for customer X in the last 30 days"*). Follows the **ESR rule:** Equality fields first, then Sort/Range fields.

### Checking If an Index Is Being Used (`explain()`)

Key fields to inspect in the output:

| Field | Expected Value |
|---|---|
| `winningPlan.stage` | `"IXSCAN"` (index scan) — `"COLLSCAN"` means index NOT used |
| `totalDocsExamined` | Should be close to `totalDocsReturned` |
| `executionTimeMillis` | Compare before and after index creation |
| `indexName` | Confirms which index the query planner selected |

In [ ]:
# ── Index creation (PyMongo) ────────────────────────────────────────────────
# from pymongo import MongoClient, ASCENDING, DESCENDING
# client = MongoClient("mongodb://localhost:27017/")
# db = client["shop"]

# Single-field index on order_date (descending — most recent first)
# db.orders.create_index([("order_date", -1)])

# Compound index (ESR rule: Equality → Sort/Range)
# db.orders.create_index(
#     [("customer_id", 1), ("order_date", -1)],
#     name="idx_customer_date"
# )

# List existing indexes
# indexes = db.orders.index_information()
# for name, details in indexes.items():
#     print(f"Index: {name} → Key: {details['key']}")

# Explain query execution plan
# from bson import ObjectId
# plan = db.orders.find(
#     {"customer_id": ObjectId("..."), "order_date": {"$gt": some_date}}
# ).explain("executionStats")
# print("Stage         :", plan["queryPlanner"]["winningPlan"]["stage"])
# print("Docs examined :", plan["executionStats"]["totalDocsExamined"])
# print("Docs returned :", plan["executionStats"]["totalDocsReturned"])
# print("Time (ms)     :", plan["executionStats"]["executionTimeMillis"])

print("Index management reference:")
cmds = [
    ("Create single-field", "db.orders.create_index([('order_date', -1)])"),
    ("Create compound",     "db.orders.create_index([('customer_id', 1), ('order_date', -1)], name='idx_customer_date')"),
    ("List indexes",        "db.orders.index_information()"),
    ("Check usage",         "db.orders.find({...}).explain('executionStats')"),
    ("Index stats",         "db.orders.aggregate([{'$indexStats': {}}])"),
]
for label, cmd in cmds:
    print(f"  {label:20s}: {cmd}")

## Question 7: Product Recommendation System

### 1. Data Structure in MongoDB

**Two primary collections:**

```
orders collection
{ _id: ObjectId, customer_id: ObjectId, order_date: Date,
  items: [ { product_id: ObjectId, product_name: str, category: str, price: float } ] }

products collection
{ _id: ObjectId, name: str, category: str, price: float, tags: [str] }
```

Embedding `items` inside `orders` avoids JOINs for the most common query: *"what did this customer buy?"*

### 2. Aggregation Pipeline — Frequently Bought Together

Uses `$unwind` → `$group` → `$sort` → `$limit` to find the most co-purchased products.

### 3. How Covariance Helps Identify Relationships

After extracting per-customer purchase frequency vectors:

| Covariance Sign | Meaning | Action |
|---|---|---|
| **Positive** | Customers who buy A also buy B frequently | Recommend together (complementary) |
| **Near zero** | No purchasing relationship | No recommendation signal |
| **Negative** | Customers who buy A tend not to buy B | They may be substitutes |

Pearson correlation (normalised covariance) powers collaborative filtering ("customers like you also bought…").

### 4. Would Sharding Impact Recommendation Performance?

| Scenario | Impact |
|---|---|
| **Sharded by `customer_id`** | ✅ Per-customer queries are fast — single shard |
| **Global aggregation across customers** | ⚠️ Scatter-gather across all shards — slow at scale |
| **Precomputed recommendations** | ✅ Batch pipeline runs nightly; real-time serving hits a fast cached collection |

In [ ]:
import pandas as pd
import numpy as np

# ── Simulate purchase data ────────────────────────────────────────────────
np.random.seed(42)
products  = ["Laptop", "Mouse", "Keyboard", "Monitor", "Headphones", "Webcam", "USB Hub"]
customers = [f"C{i:03d}" for i in range(1, 51)]

# Purchase frequency matrix: rows=customers, cols=products
freq_matrix = np.random.poisson(lam=[3, 2, 2, 1, 1.5, 1, 0.5], size=(50, 7))
df_freq = pd.DataFrame(freq_matrix, index=customers, columns=products)

print("Purchase frequency matrix (first 5 customers):")
print(df_freq.head())

# ── Co-purchase covariance matrix ────────────────────────────────────────
cov_matrix  = df_freq.cov()
corr_matrix = df_freq.corr()

print("
Correlation matrix between products:")
print(corr_matrix.round(3))

# ── Find top complementary pairs ─────────────────────────────────────────
import itertools
pairs = []
for p1, p2 in itertools.combinations(products, 2):
    pairs.append({"Product A": p1, "Product B": p2, "Correlation": corr_matrix.loc[p1, p2]})

pairs_df = pd.DataFrame(pairs).sort_values("Correlation", ascending=False)
print("
Top complementary product pairs (recommend together):")
print(pairs_df.head(5).to_string(index=False))

print("
Substitute product pairs (negative correlation):")
print(pairs_df.tail(3).to_string(index=False))

In [ ]:
# ── MongoDB Aggregation Pipeline — frequently bought together ────────────
# (Commented — requires a live MongoDB connection)

aggregation_pipeline = [
    {"$unwind": "$items"},
    {"$group": {
        "_id": "$_id",
        "products": {"$push": "$items.product_id"}
    }},
    {"$unwind": "$products"},
    {"$group": {
        "_id": "$products",
        "order_count": {"$sum": 1}
    }},
    {"$sort": {"order_count": -1}},
    {"$limit": 10}
]

import json
print("Aggregation pipeline (run via db.orders.aggregate([...])):")
print(json.dumps(aggregation_pipeline, indent=2))

# result = list(db.orders.aggregate(aggregation_pipeline))

## Question 8: Bayes' Theorem — Churn Probability Calculation

### Given Information

| Variable | Value |
|---|---|
| P(Churn) | 0.05 (5% of customers churn) |
| P(Not Churn) | 0.95 |
| P(Declining Frequency \| Churn) | 0.70 |
| P(Declining Frequency \| Not Churn) | 0.15 |

### Applying Bayes' Theorem

We want **P(Churn | Declining Frequency)**.

**Step 1: Total Probability of Declining Frequency**

`P(Declining) = P(Declining|Churn)×P(Churn) + P(Declining|Not Churn)×P(Not Churn)`  
`P(Declining) = (0.70 × 0.05) + (0.15 × 0.95) = 0.035 + 0.1425 = 0.1775`

**Step 2: Apply Bayes' Formula**

`P(Churn | Declining) = (0.70 × 0.05) / 0.1775 = 0.035 / 0.1775 ≈ 0.1972 (≈ 19.72%)`

### Interpretation

A customer showing declining order frequency has approximately a **19.7% probability of churning** — nearly **4× higher** than the baseline rate of 5%. However, ~80% of declining-frequency customers will NOT churn. The business should use this posterior probability to **prioritize retention interventions** rather than automatically classifying all such customers as churners.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# ── Bayes' Theorem — Churn Calculation ──────────────────────────────────────
p_churn           = 0.05
p_not_churn       = 1 - p_churn
p_declining_churn     = 0.70
p_declining_no_churn  = 0.15

# Step 1: Total probability of declining frequency
p_declining = (p_declining_churn * p_churn) + (p_declining_no_churn * p_not_churn)

# Step 2: Bayes' formula
p_churn_given_declining = (p_declining_churn * p_churn) / p_declining

print("=== Bayes' Theorem — Churn Probability ===")
print(f"
Step 1 — P(Declining):")
print(f"  = P(Declining|Churn) × P(Churn) + P(Declining|Not Churn) × P(Not Churn)")
print(f"  = ({p_declining_churn} × {p_churn}) + ({p_declining_no_churn} × {p_not_churn})")
print(f"  = {p_declining_churn*p_churn} + {p_declining_no_churn*p_not_churn}")
print(f"  = {p_declining:.4f}")
print(f"
Step 2 — P(Churn | Declining):")
print(f"  = (P(Declining|Churn) × P(Churn)) / P(Declining)")
print(f"  = ({p_declining_churn} × {p_churn}) / {p_declining:.4f}")
print(f"  = {p_declining_churn*p_churn} / {p_declining:.4f}")
print(f"  = {p_churn_given_declining:.4f}  ({p_churn_given_declining*100:.2f}%)")
print(f"
Uplift over baseline: {p_churn_given_declining/p_churn:.1f}×")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Bar comparison
labels = ['Baseline
P(Churn)', 'Posterior
P(Churn | Declining)']
values = [p_churn, p_churn_given_declining]
colors = ['#90CAF9', '#EF5350']
bars = axes[0].bar(labels, values, color=colors, width=0.45, edgecolor='white')
for bar, val in zip(bars, values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.2%}', ha='center', fontsize=12, fontweight='bold')
axes[0].set_title("Churn Probability: Baseline vs Posterior", fontsize=13)
axes[0].set_ylabel("Probability")
axes[0].set_ylim(0, 0.28)

# Pie: composition of declining-frequency customers
pie_labels = ['Will NOT Churn', 'Will Churn']
pie_vals   = [1 - p_churn_given_declining, p_churn_given_declining]
axes[1].pie(pie_vals, labels=pie_labels, autopct='%1.1f%%',
            colors=['#A5D6A7', '#EF9A9A'], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title("Composition of Declining-Frequency
Customers", fontsize=13)

plt.tight_layout()
plt.savefig('churn_bayes.png', dpi=150, bbox_inches='tight')
plt.show()

## Question 9: Central Limit Theorem (CLT)

### 1. Why CLT Allows Us to Use Normal Approximation

The **Central Limit Theorem** states: if you take sufficiently large random samples from **any** population (regardless of its underlying distribution — uniform, exponential, right-skewed, etc.), the distribution of **sample means** will approximate a **normal distribution** as sample size increases — typically when **n ≥ 30**.

This means we do not need to assume the original data is normally distributed. As long as we work with sample means from large enough samples, we can apply z-scores, confidence intervals, and p-values.

*Example:* Customer spending is right-skewed, but if we take samples of 100 customers and compute each sample's mean, those sample means will be approximately normally distributed.

### 2. Why CLT is Important in A/B Testing

CLT makes A/B testing statistically rigorous:
- Metric differences between groups follow a normal distribution (via CLT), allowing computation of z-scores and p-values.
- We can calculate **confidence intervals** around the estimated effect size — e.g., *"new button increases conversion rate by 2.3% ± 0.8% at 95% confidence."*
- Without CLT, we could not apply parametric tests to metrics like revenue or session duration that are not raw-normally distributed.

### 3. What Happens If Sample Size Is Very Small (n < 30)

| Problem | Effect |
|---|---|
| CLT approximation breaks down | Sample means no longer reliably normal |
| Use t-distribution, not z | Heavier tails account for uncertainty from small samples |
| Large standard errors | Wide confidence intervals; hard to detect true effects |
| Low statistical power | Frequent false negatives; sensitive to outliers |
| **Solution** | Run power analysis before the experiment; target ≥ 80% statistical power |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# Underlying population: right-skewed (log-normal)
population = np.random.lognormal(mean=7, sigma=1.5, size=100_000)

sample_sizes = [5, 10, 30, 100]
fig, axes = plt.subplots(1, len(sample_sizes) + 1, figsize=(18, 4))

# Show original population
axes[0].hist(population, bins=80, color='coral', edgecolor='white', alpha=0.8)
axes[0].set_title('Original Population
(Right-Skewed, n=100,000)', fontsize=10)
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Count')

# Show distribution of sample means for each sample size
for ax, n in zip(axes[1:], sample_sizes):
    sample_means = [np.mean(np.random.choice(population, size=n, replace=False))
                    for _ in range(2000)]

    ax.hist(sample_means, bins=50, color='steelblue', edgecolor='white', alpha=0.8, density=True)

    # Overlay theoretical normal
    mu, sigma = np.mean(sample_means), np.std(sample_means)
    x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
    ax.plot(x, stats.norm.pdf(x, mu, sigma), 'r-', linewidth=2, label='Normal fit')

    sw_stat, sw_p = stats.shapiro(np.random.choice(sample_means, 100))
    ax.set_title(f'Sample Means (n={n})
Shapiro p={sw_p:.3f}', fontsize=10)
    ax.set_xlabel('Sample Mean')
    ax.legend(fontsize=8)

plt.suptitle('Central Limit Theorem — Distribution of Sample Means', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('clt_demo.png', dpi=150, bbox_inches='tight')
plt.show()
print("As n increases, sample means converge to a normal distribution regardless of population shape.")

## Question 10: Hypothesis Testing — Average Monthly Spending Claim

### 1. How to Statistically Test the Claim

Use a **One-Sample t-test** (z-test if population σ is known and n is large):

1. Collect a random sample of customer monthly spending (e.g., n = 200).
2. Compute the sample mean (x̄) and sample standard deviation (s).
3. Calculate the test statistic: `t = (x̄ − μ₀) / (s / √n)` where μ₀ = 5000.
4. Determine the p-value from the t-distribution with n−1 degrees of freedom.
5. Compare p-value to significance level α (typically 0.05). If p < 0.05, reject H₀.

### 2. Null and Alternative Hypothesis

- **H₀ (Null):** The average monthly spending equals ₹5000. `H₀: μ = 5000` — assumed correct unless sufficient evidence rejects it.
- **H₁ (Alternative):** The average monthly spending is NOT equal to ₹5000. `H₁: μ ≠ 5000` — two-tailed test (we care if the true mean is either higher or lower).

### 3. Type I vs Type II Error

| | Definition | Our Example | Business Consequence |
|---|---|---|---|
| **Type I (False Positive)** | Rejecting H₀ when it's actually true | Conclude spending ≠ ₹5000 when it is | Taking action (changing strategy) based on a false alarm |
| **Type II (False Negative)** | Failing to reject H₀ when it's actually false | Conclude spending = ₹5000 when it's actually ₹3000 | Missing a real problem — forecasts, planning, and budgets all wrong |

**Which is more dangerous?** In this spending scenario, a **Type II error** is arguably more dangerous. If the true average has silently dropped to ₹3,000 but goes undetected, inventory planning, revenue forecasts, and financial projections will all be based on incorrect assumptions, causing compounding business damage over time.

> The balance between Type I and Type II error is controlled by the **significance level (α)** and **statistical power (1 − β)**. Lowering α reduces Type I errors but increases Type II errors. Set these thresholds deliberately based on the cost of each mistake in your specific context.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)

# ── Simulate customer spending sample ────────────────────────────────────────
claimed_mean = 5000
n = 200

# Scenario A: true mean ≈ 5000 (H0 true)
sample_a = np.random.normal(loc=4900, scale=1500, size=n)

# Scenario B: true mean ≈ 3000 (H0 false — Type II risk)
sample_b = np.random.normal(loc=3200, scale=1400, size=n)

def run_ttest(sample, mu0, label):
    xbar = np.mean(sample)
    s    = np.std(sample, ddof=1)
    se   = s / np.sqrt(len(sample))
    t_stat, p_value = stats.ttest_1samp(sample, popmean=mu0)

    print(f"
=== {label} ===")
    print(f"  Sample mean   : ₹{xbar:,.2f}")
    print(f"  Sample std    : ₹{s:,.2f}")
    print(f"  Std Error     : ₹{se:,.2f}")
    print(f"  t-statistic   : {t_stat:.4f}")
    print(f"  p-value       : {p_value:.6f}")
    decision = "REJECT H₀" if p_value < 0.05 else "FAIL TO REJECT H₀"
    print(f"  Decision (α=0.05): {decision}")
    return t_stat, p_value

t_a, p_a = run_ttest(sample_a, claimed_mean, "Scenario A — True mean ≈ ₹5000")
t_b, p_b = run_ttest(sample_b, claimed_mean, "Scenario B — True mean ≈ ₹3000")

# ── Visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, sample, t, p, label, color in zip(
    axes,
    [sample_a, sample_b],
    [t_a, t_b],
    [p_a, p_b],
    ["Scenario A (True mean ≈ ₹5000)", "Scenario B (True mean ≈ ₹3000)"],
    ['steelblue', 'coral']
):
    ax.hist(sample, bins=30, color=color, edgecolor='white', alpha=0.7)
    ax.axvline(claimed_mean,     color='black', linestyle='--', linewidth=2, label=f'Claimed μ = ₹{claimed_mean:,}')
    ax.axvline(np.mean(sample),  color='red',   linestyle='-',  linewidth=2, label=f'Sample x̄ = ₹{np.mean(sample):,.0f}')
    ax.set_title(f'{label}
t={t:.3f}, p={p:.4f}', fontsize=10)
    ax.set_xlabel('Monthly Spending (₹)')
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=9)

plt.suptitle('One-Sample t-Test: Is Average Spending = ₹5,000?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('hypothesis_test.png', dpi=150, bbox_inches='tight')
plt.show()